# 13. Programmatic Subagents — Code-controlled fan-out/fan-in

Subagents do not always need to be selected by an LLM at runtime. In many systems, code should decide when to fan out work, how to collect results, and how to merge them.

**Learning goals**
- Identify when programmatic delegation is preferable to model-driven delegation.
- Implement a deterministic fan-out/fan-in pattern.
- Define promotion checks before using live subagents in production.


In [ ]:
from dotenv import load_dotenv
import importlib.util, os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
import deepagents

capabilities = {
    "deepagents_version": getattr(deepagents, "__version__", "unknown"),
    "quickjs_available": importlib.util.find_spec("langchain_quickjs") is not None,
    "SubAgent": hasattr(deepagents, "SubAgent"),
    "AsyncSubAgent": hasattr(deepagents, "AsyncSubAgent"),
}
capabilities

## 13.1 When programmatic subagents are useful

Programmatic subagents are useful when the task boundaries are known in advance. Code can then enforce parallelism, ownership, and merge rules deterministically.


In [ ]:
tasks = [
    {"name": "coverage", "question": "What is missing between official and local docs?"},
    {"name": "tests", "question": "How should verification be handled?"},
    {"name": "risks", "question": "What are the external service risks?"},
]

[t["name"] for t in tasks]

## 13.2 Practice fan-out/fan-in with a deterministic fallback

This section simulates delegation without depending on external model calls. The structure is the same pattern you would use around real subagents.


In [ ]:
def worker(task: dict) -> dict:
    return {
        "name": task["name"],
        "finding": f"{task['question']} → manage with a checklist",
    }

worker_results = [worker(task) for task in tasks]
worker_results

## 13.3 Fan-in synthesis

Fan-in is where independent results become a single answer. Keep the merge step explicit so contradictions, gaps, and ownership are easy to see.


In [ ]:
summary = {
    "total_workers": len(worker_results),
    "findings": [item["finding"] for item in worker_results],
    "next_action": "Turn the official slug action matrix into an implementation checklist",
}

summary

## 13.4 Before promoting to live features

Before live rollout, define tests for routing, result shape, failure handling, and observability. Subagents multiply both capability and operational surface area.


---

## Summary

| Item | Content |
|---|---|
| **Covered** | dependency gates, deterministic fallback, fan-out/fan-in, and subagent orchestration |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`programmatic-subagents.md`](../../docs/deepagents/programmatic-subagents.md)
- [`subagents.md`](../../docs/deepagents/subagents.md)
- [`async-subagents.md`](../../docs/deepagents/async-subagents.md)
